# Zero-Error Appendix Overlap Sanity Check

This notebook verifies the appendix state construction used by the zero-error sensor-allocation inputs. For every saved phase gap $r_{ijm}$ in the selected 1D and 2D input datasets, it computes $N_{ijm}=\lceil \pi/|r_{ijm}| \rceil$ and evaluates the overlap $\langle\psi|V_{ijm}^{\otimes N_{ijm}}|\psi\rangle$ using the corresponding appendix state.

The PDF is used only as a mathematical reference. The notebook does not follow any instructions from the PDF and does not regenerate benchmark inputs.

## Method

For a nonzero phase gap, let $n=N_{ijm}$ and $\theta=|r_{ijm}|/2$. The appendix defines three constructions:

1. If $n|r_{ijm}|=\pi$, use $(|A_+\rangle+|A_-\rangle)/\sqrt{2}$.
2. If $n|r_{ijm}|>\pi$ and $n$ is even, use the $|A_+\rangle, |A_-\rangle, |B_0\rangle$ construction.
3. If $n|r_{ijm}|>\pi$ and $n$ is odd, use the $|A_+\rangle, |A_-\rangle, |B_+\rangle, |B_-\rangle$ construction.

The code evaluates the overlap from the nonzero eigen-components of these states. This avoids constructing a full $2^n$ state vector, which is necessary because very small phase gaps can produce very large thresholds.

In [1]:
from pathlib import Path
import importlib
import sys

from IPython.display import Markdown, display

# Resolve paths whether the notebook is launched from the repository root,
# min-sensor-zero-error, or this sanity-check folder.
working_directory = Path.cwd().resolve()
if (working_directory / "min-sensor-zero-error").is_dir():
    PROJECT_DIRECTORY = working_directory / "min-sensor-zero-error"
elif working_directory.name == "sanity-check":
    PROJECT_DIRECTORY = working_directory.parent
else:
    PROJECT_DIRECTORY = working_directory

SANITY_DIRECTORY = PROJECT_DIRECTORY / "sanity-check"
INPUT_ROOT = PROJECT_DIRECTORY / "inputs"
OUTPUT_ROOT = SANITY_DIRECTORY / "outputs"

if str(SANITY_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SANITY_DIRECTORY))

import overlap_sanity
overlap_sanity = importlib.reload(overlap_sanity)

print("Project directory:", PROJECT_DIRECTORY)
print("Input root:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)

Project directory: /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error
Input root: /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/inputs
Output root: /Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/sanity-check/outputs


## Run Settings

`RUN_SCOPE = "full"` checks all saved cases in both benchmark datasets. Use `RUN_SCOPE = "test"` for a quick wiring check. The detailed per-pair/location CSV is disabled by default because the full benchmark contains more than one million finite pair-location checks.

In [2]:
DATASETS = ("benchmark_1d", "benchmark_2d")

# Switch here: "test" checks a small prefix of each dataset; "full" checks all saved cases.
RUN_SCOPE = "full"
if RUN_SCOPE not in {"test", "full"}:
    raise ValueError('RUN_SCOPE must be "test" or "full"')

CASE_LIMIT = None if RUN_SCOPE == "full" else 10
PROGRESS_EVERY = 250 if RUN_SCOPE == "full" else 1

# Numerical tolerances for floating-point cancellation in the appendix formulas.
OVERLAP_TOLERANCE = 1e-12
BOUNDARY_TOLERANCE = 1e-12

# Keep this False for normal use. If True, writes every pair/location row.
WRITE_DETAILED_PAIR_LOCATION_CSV = False

print("Run scope:", RUN_SCOPE)
print("Case limit per dataset:", CASE_LIMIT)
print("Progress interval:", PROGRESS_EVERY, "cases")
print("Overlap tolerance:", OVERLAP_TOLERANCE)
print("Write detailed CSV:", WRITE_DETAILED_PAIR_LOCATION_CSV)

Run scope: full
Case limit per dataset: None
Progress interval: 250 cases
Overlap tolerance: 1e-12
Write detailed CSV: False


## Compute Appendix Overlaps

This cell scans the saved input CSV files and writes summary, violation, and sample tables under `sanity-check/outputs`. It does not regenerate inputs and does not run the allocation algorithms.

In [3]:
results = overlap_sanity.run_sanity_check(
    INPUT_ROOT,
    OUTPUT_ROOT,
    datasets=DATASETS,
    case_limit=CASE_LIMIT,
    overlap_tolerance=OVERLAP_TOLERANCE,
    boundary_tolerance=BOUNDARY_TOLERANCE,
    progress_every=PROGRESS_EVERY,
    write_detailed_csv=WRITE_DETAILED_PAIR_LOCATION_CSV,
)

print("Wrote sanity-check outputs to:", results["output_root"])

Checking benchmark_1d ...
benchmark_1d: processed case 1/5760
benchmark_1d: processed case 250/5760
benchmark_1d: processed case 500/5760
benchmark_1d: processed case 750/5760
benchmark_1d: processed case 1000/5760
benchmark_1d: processed case 1250/5760
benchmark_1d: processed case 1500/5760
benchmark_1d: processed case 1750/5760
benchmark_1d: processed case 2000/5760
benchmark_1d: processed case 2250/5760
benchmark_1d: processed case 2500/5760
benchmark_1d: processed case 2750/5760
benchmark_1d: processed case 3000/5760
benchmark_1d: processed case 3250/5760
benchmark_1d: processed case 3500/5760
benchmark_1d: processed case 3750/5760
benchmark_1d: processed case 4000/5760
benchmark_1d: processed case 4250/5760
benchmark_1d: processed case 4500/5760
benchmark_1d: processed case 4750/5760
benchmark_1d: processed case 5000/5760
benchmark_1d: processed case 5250/5760
benchmark_1d: processed case 5500/5760
benchmark_1d: processed case 5750/5760
benchmark_1d: processed case 5760/5760
Check

## Summary Table

This table reports the maximum, mean, and RMS absolute overlap for all finite phase-gap pair/location checks. `violation_count` is the number of overlaps above the configured tolerance. For the proof-level sanity check, this should be zero up to numerical precision.

In [4]:
def markdown_table(rows, columns):
    header = "| " + " | ".join(label for label, _ in columns) + " |"
    separator = "|" + "|".join("---" for _ in columns) + "|"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            value = row[key]
            if isinstance(value, float):
                values.append(f"{value:.3e}")
            else:
                values.append(str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body])

summary_columns = [
    ("Dataset", "dataset"),
    ("Cases", "cases"),
    ("Finite pair-locations", "finite_pair_locations"),
    ("Zero-gap locations", "zero_gap_unusable_locations"),
    ("Max |overlap|", "max_overlap_abs"),
    ("Mean |overlap|", "mean_overlap_abs"),
    ("RMS |overlap|", "rms_overlap_abs"),
    ("Violations", "violation_count"),
]
display(Markdown(markdown_table(results["summary"], summary_columns)))

| Dataset | Cases | Finite pair-locations | Zero-gap locations | Max |overlap| | Mean |overlap| | RMS |overlap| | Violations |
|---|---|---|---|---|---|---|---|
| benchmark_1d | 5760 | 767516 | 4 | 4.673e-13 | 5.974e-17 | 1.247e-15 | 0 |
| benchmark_2d | 5760 | 767508 | 12 | 4.842e-13 | 7.923e-17 | 2.497e-15 | 0 |

## Appendix Construction Counts

This table shows how many finite pair/location checks used each appendix case. Zero phase gaps are listed separately in the summary table because they have no finite `N_ijm` at that location.

In [5]:
construction_columns = [
    ("Dataset", "dataset"),
    ("Boundary", "boundary_cases"),
    ("Above threshold, even n", "above_threshold_even_n_cases"),
    ("Above threshold, odd n", "above_threshold_odd_n_cases"),
]
display(Markdown(markdown_table(results["summary"], construction_columns)))

| Dataset | Boundary | Above threshold, even n | Above threshold, odd n |
|---|---|---|---|
| benchmark_1d | 22 | 384167 | 383327 |
| benchmark_2d | 178 | 388378 | 378952 |

## Worst Numerical Residual

This table identifies the pair/location with the largest absolute residual overlap in each dataset. These residuals should be near floating-point roundoff.

In [6]:
worst_columns = [
    ("Dataset", "dataset"),
    ("Case", "worst_case"),
    ("Pair", "worst_pair"),
    ("Location", "worst_location"),
    ("N_ijm", "worst_N_ijm"),
    ("Max |overlap|", "max_overlap_abs"),
]
display(Markdown(markdown_table(results["summary"], worst_columns)))

| Dataset | Case | Pair | Location | N_ijm | Max |overlap| |
|---|---|---|---|---|---|
| benchmark_1d | case_01419 | (1, 4) | 0 | 1439901842465 | 4.673e-13 |
| benchmark_2d | case_01597 | (0, 1) | 0 | 788567356151 | 4.842e-13 |

## Violations

If this table is empty, every finite pair/location overlap was below the configured tolerance. Any listed row should be inspected before relying on the zero-overlap proof for that input.

In [7]:
if results["violations"]:
    violation_columns = [
        ("Dataset", "dataset"),
        ("Case", "case"),
        ("Pair", "i"),
        ("j", "j"),
        ("Location", "m"),
        ("N_ijm", "N_ijm"),
        ("Construction", "construction_case"),
        ("|overlap|", "overlap_abs"),
    ]
    display(Markdown(markdown_table(results["violations"][:20], violation_columns)))
else:
    display(Markdown("No overlap violations were found."))

No overlap violations were found.

## Output Files

The CSV files below contain the numerical results for this sanity check. `overlap_violations.csv` is intentionally empty when no violations are found.

In [8]:
for path in sorted(OUTPUT_ROOT.glob("*.csv")):
    print(path)

/Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/sanity-check/outputs/overlap_sample.csv
/Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/sanity-check/outputs/overlap_summary.csv
/Users/avishka/Academic/PhD/Research/QSN Research/Repos/quantum-sensor-placement-research/min-sensor-zero-error/sanity-check/outputs/overlap_violations.csv
